# TaskGen Indonesia — Generator Dataset dengan Gemini API

Notebook ini menjalankan versi Rust dari TaskGen. Atur **jumlah tugas, kategori/topik, level, dan model** sebelum menjalankan. Output disimpan ke Google Drive.

> API key jangan ditulis di sel kode atau dikirim ke GitHub. Gunakan Colab Secrets `GEMINI_API_KEY` jika memungkinkan.

In [ ]:
# 1) Hubungkan repositori GitHub yang sudah kamu push
import os, pathlib, subprocess, shutil, getpass, sys

repo_url = input("URL repositori GitHub TaskGen Indonesia yang sudah kamu push: ").strip()
if not repo_url:
    raise ValueError("URL repositori wajib diisi.")
repo_dir = pathlib.Path("/content/taskgen-indonesia")
if repo_dir.exists():
    shutil.rmtree(repo_dir)
subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)
print("Repositori siap:", repo_dir)

In [ ]:
# 2) Pasang Rust/Cargo jika belum tersedia, lalu kompilasi program
if shutil.which("cargo") is None:
    print("Memasang Rust toolchain di runtime Colab...")
    subprocess.run(
        ["bash", "-lc", "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y"],
        check=True,
    )
    os.environ["PATH"] = os.environ.get("PATH", "") + ":/root/.cargo/bin"

subprocess.run(["cargo", "build", "--release"], cwd=repo_dir, check=True)
binary = repo_dir / "target" / "release" / "taskgen"
if not binary.exists():
    raise FileNotFoundError("Binary TaskGen tidak ditemukan setelah kompilasi.")
print("Kompilasi selesai:", binary)

In [ ]:
# 3) Siapkan Gemini API key dan Google Drive
from google.colab import drive
from google.colab import userdata
from getpass import getpass

try:
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    api_key = None
if not api_key:
    api_key = getpass("Masukkan Gemini API key (tidak akan ditampilkan): ").strip()
if not api_key:
    raise ValueError("Gemini API key kosong.")
os.environ["GEMINI_API_KEY"] = api_key

drive.mount("/content/drive")
drive_dir = pathlib.Path("/content/drive/MyDrive/TaskGen-Indonesia")
drive_dir.mkdir(parents=True, exist_ok=True)
print("Folder output:", drive_dir)

In [ ]:
# 4) Lihat model yang didukung API, lalu pilih model untuk run ini
import requests

models_url = "https://generativelanguage.googleapis.com/v1beta/models"
resp = requests.get(models_url, headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]}, timeout=30)
if not resp.ok:
    print(f"Peringatan: daftar model tidak bisa diambil ({resp.status_code}). Kamu tetap bisa mengetik ID model secara manual.")
    model_items = []
else:
    model_items = resp.json().get("models", [])
text_models = []
for item in model_items:
    methods = item.get("supportedGenerationMethods", [])
    name = item.get("name", "").replace("models/", "", 1)
    if name and "generateContent" in methods:
        text_models.append((name, item.get("displayName", name)))
text_models.sort(key=lambda x: x[0])

print("Model yang mendukung generateContent:")
for i, (name, display) in enumerate(text_models, 1):
    print(f"{i}. {display} — {name}")

default_model = "gemini-3.5-flash-lite"
model_input = input(f"Pilih nomor model atau ketik ID model sendiri [{default_model}]: ").strip()
if not model_input:
    model = default_model
elif model_input.isdigit() and 1 <= int(model_input) <= len(text_models):
    model = text_models[int(model_input)-1][0]
else:
    model = model_input
print("Model untuk run ini:", model)
print("Catatan: model yang tersedia belum tentu punya kuota gratis pada akunmu.")

In [ ]:
# 5) Atur jumlah tugas, topik, level, dan opsi run
from datetime import datetime

print("Kategori: 1=Matematika, 2=Pemrograman, 3=Sains, 4=Ilmu komputer, 5=Kreativitas/Penulisan, 6=Percakapan")
count = int(input("Jumlah tugas yang ingin dibuat [100]: ").strip() or "100")
if count < 1:
    raise ValueError("Jumlah tugas harus minimal 1.")
cat_input = input("Pilih kategori dengan angka dipisahkan koma, atau ketik 'semua' [semua]: ").strip().lower() or "semua"
category_map = {"1":"math", "2":"coding", "3":"science", "4":"cs", "5":"creative", "6":"conversation"}
if cat_input in ("semua", "all", "*"):
    selected = list(category_map.values())
else:
    selected = []
    for part in cat_input.split(","):
        part = part.strip()
        if part not in category_map:
            raise ValueError(f"Pilihan kategori tidak dikenal: {part}")
        if category_map[part] not in selected:
            selected.append(category_map[part])
if not selected:
    raise ValueError("Pilih minimal satu kategori.")
weight = round(1.0 / len(selected), 6)
weights = [weight] * len(selected)
weights[-1] = round(1.0 - sum(weights[:-1]), 6)
distribution = ",".join(f"{cat}={w}" for cat, w in zip(selected, weights))

level_input = input("Level 1–10, atau 'campuran' untuk distribusi bawaan [campuran]: ").strip().lower() or "campuran"
if level_input in ("campuran", "campur", "mixed", "all"):
    difficulty_args = []
else:
    level = int(level_input)
    if not 1 <= level <= 10:
        raise ValueError("Level harus antara 1 sampai 10, atau 'campuran'.")
    difficulty_args = ["--difficulty", f"{level}=1.0"]

workers_input = input("Jumlah worker paralel [2; disarankan 1–2 untuk kuota gratis]: ").strip() or "2"
workers = int(workers_input)
if workers < 1:
    raise ValueError("Workers harus minimal 1.")
dedup_input = input("Hapus duplikat setelah generasi? (y/N): ").strip().lower()

run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = drive_dir / run_stamp
run_dir.mkdir(parents=True, exist_ok=True)
output_path = run_dir / "dataset.jsonl"
print("Ringkasan:")
print(" - Jumlah tugas:", count)
print(" - Kategori:", ", ".join(selected))
print(" - Level:", level_input)
print(" - Model:", model)
print(" - Worker:", workers)
print(" - Output:", output_path)

In [ ]:
# 6) Jalankan generasi; setiap tugas yang berhasil langsung ditulis ke Google Drive
command = [
    str(binary),
    "--count", str(count),
    "--model", model,
    "--distribution", distribution,
    "--workers", str(workers),
    "--output", str(output_path),
]
command.extend(difficulty_args)
if dedup_input in ("y", "yes", "ya"):
    command.append("--dedup")
print("Memulai TaskGen. API key tidak ditampilkan.")
result = subprocess.run(command, cwd=repo_dir, env=os.environ.copy())
print("Exit code:", result.returncode)
print("File output:", output_path)
print("README output:", output_path.parent / "README.md")
if output_path.exists():
    print("Ukuran file:", output_path.stat().st_size, "bytes")
    print("Jumlah baris saat ini:", sum(1 for line in output_path.open(encoding="utf-8") if line.strip()))
if result.returncode != 0:
    print("Run berakhir dengan error. Periksa pesan di atas; hasil yang sudah ditulis tetap tersimpan di Drive.")

## Catatan penggunaan

- Mulai uji coba dengan 10–20 tugas dan `workers=1` atau `2` sebelum membuat dataset besar.
- Program mempertahankan satu model untuk satu run. Jika kena `429`, program menunggu lalu mencoba ulang. Jika limit tetap habis, pilih model lain saat run berikutnya; perpindahan otomatis tidak diaktifkan.
- Hasil disimpan ke Google Drive, sedangkan kode sumber tetap berada di GitHub.
- Jangan simpan API key di file notebook atau repositori.